In [ ]:
import os
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("ECommerce_RFM_ML_Pipeline")
    .config("spark.sql.parquet.compression.codec", "snappy")
    .getOrCreate()
)

curated_input_path = "../data/curated/cleaned_orders.parquet"

# Load cleaned Parquet file from Notebook 1
df_cleaned = spark.read.parquet(curated_input_path)
df_cleaned.createOrReplaceTempView("cleaned_orders")

print("Curated data loaded and 'cleaned_orders' view created.")
df_cleaned.show(5)

In [ ]:
rfm_sql = """
WITH max_date_cte AS (
    SELECT MAX(Order_Timestamp) AS max_dataset_date FROM cleaned_orders
)
SELECT 
    c.Customer_ID,
    COUNT(c.Order_ID) AS frequency,
    ROUND(SUM(c.Order_Amount), 2) AS monetary,
    DATEDIFF(m.max_dataset_date, MAX(c.Order_Timestamp)) AS recency,
    AVG(c.Returned_Flag) AS return_rate
FROM cleaned_orders c
CROSS JOIN max_date_cte m
GROUP BY c.Customer_ID, m.max_dataset_date
"""

df_rfm = spark.sql(rfm_sql)
df_rfm.createOrReplaceTempView("rfm_features")

df_rfm.show(5)
print(f"Total Unique Customers: {df_rfm.count()}")

In [ ]:
import mlflow
import mlflow.spark
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import ClusteringEvaluator

# 1. Define MLflow Experiment Workspace
mlflow.set_experiment("/Shared/Customer_Segmentation_MLOps")
with mlflow.start_run(run_name="kmeans_rfm_clustering") as run:

    # --- Log Hyperparameters ---
    k_clusters = 3
    seed_val = 42
    
    mlflow.log_param("k_clusters", k_clusters)
    mlflow.log_param("seed", seed_val)
    mlflow.log_param("distance_measure", "euclidean")

    # --- Build Pipeline Stages ---
    assembler = VectorAssembler(
        inputCols=["recency", "frequency", "monetary"], 
        outputCol="raw_features"
    )
    scaler = StandardScaler(
        inputCol="raw_features", 
        outputCol="scaled_features", 
        withStd=True, 
        withMean=True
    )
    kmeans = KMeans(
        featuresCol="scaled_features", 
        predictionCol="cluster_id", 
        k=k_clusters, 
        seed=seed_val
    )
    pipeline = Pipeline(stages=[assembler, scaler, kmeans])

    # --- Fit Pipeline & Predict ---
    model = pipeline.fit(df_rfm)
    df_clustered = model.transform(df_rfm)

    # --- Evaluate Model Quality ---
    evaluator = ClusteringEvaluator(
        featuresCol="scaled_features", 
        predictionCol="cluster_id", 
        metricName="silhouette"
    )
    silhouette_score = evaluator.evaluate(df_clustered)

    # --- Log Metrics & Model Artifacts ---
    mlflow.log_metric("silhouette_score", silhouette_score)
    mlflow.spark.log_model(model, "kmeans_rfm_model")
    print(f"MLflow Run ID: {run.info.run_id}")
    print(f"Logged Silhouette Score: {silhouette_score:.4f}")

In [ ]:
df_clustered.createOrReplaceTempView("customer_clusters")

cluster_profile_sql = """
SELECT 
    cluster_id,
    COUNT(Customer_ID) AS total_customers,
    ROUND(AVG(recency), 1) AS avg_recency_days,
    ROUND(AVG(frequency), 1) AS avg_frequency,
    ROUND(AVG(monetary), 2) AS avg_monetary_spend,
    ROUND(AVG(return_rate), 2) AS avg_return_rate
FROM customer_clusters
GROUP BY cluster_id
ORDER BY avg_monetary_spend DESC
"""

df_profiles = spark.sql(cluster_profile_sql)
df_profiles.show()

In [ ]:
# Aggregate mean metrics per cluster in Spark SQL
df_profiles_pd = spark.sql("""
    SELECT 
        cluster_id,
        ROUND(AVG(recency), 1) AS avg_recency,
        ROUND(AVG(frequency), 1) AS avg_frequency,
        ROUND(AVG(monetary), 2) AS avg_monetary
    FROM customer_clusters
    GROUP BY cluster_id
    ORDER BY cluster_id
""").toPandas()

# Plot Average Monetary Spend per Cluster
plt.figure(figsize=(8, 5))
barplot = sns.barplot(
    data=df_profiles_pd,
    x="cluster_id",
    y="avg_monetary",
    palette="viridis"
)
plt.title("Average Monetary Spend by Customer Cluster", fontsize=14)
plt.xlabel("Cluster ID")
plt.ylabel("Average Spend ($)")

# Annotate bars with values
for p in barplot.patches:
    barplot.annotate(
        f"${p.get_height():,.2f}",
        (p.get_x() + p.get_width() / 2.0, p.get_height()),
        ha="center",
        va="center",
        xytext=(0, 9),
        textcoords="offset points",
    )

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Convert clustered Spark DataFrame to Pandas for local visualization
df_pd = df_clustered.select(
    "recency", "frequency", "monetary", "cluster_id"
).toPandas()

# Set plot style
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Monetary vs Recency by Cluster
sns.scatterplot(
    data=df_pd,
    x="recency",
    y="monetary",
    hue="cluster_id",
    palette="viridis",
    s=70,
    ax=axes[0],
)
axes[0].set_title("Customer Segments: Monetary vs Recency", fontsize=14)
axes[0].set_xlabel("Recency (Days Since Last Purchase)")
axes[0].set_ylabel("Monetary Spend ($)")

# Plot 2: Frequency vs Recency by Cluster
sns.scatterplot(
    data=df_pd,
    x="recency",
    y="frequency",
    hue="cluster_id",
    palette="viridis",
    s=70,
    ax=axes[1],
)
axes[1].set_title("Customer Segments: Frequency vs Recency", fontsize=14)
axes[1].set_xlabel("Recency (Days Since Last Purchase)")
axes[1].set_ylabel("Total Frequency (Order Count)")

plt.tight_layout()
plt.show()

In [ ]:
final_output_path = "../data/curated/customer_segments.parquet"

df_clustered.write \
    .mode("overwrite") \
    .parquet(final_output_path)

print(f"Final customer segments saved to: {final_output_path}")